In [ ]:
import sys
import threading
import webbrowser
import time

# Make sure the code directory is on the path
sys.path.insert(0, r"C:\Users\domenicbrooks\OneDrive - Research Triangle Institute\CIROH Flash Flooding\code")

# Reload utils FIRST so the updated discharge_to_stage is in memory,
# then reload dashboard_mrms so its 'from utils import ...' picks up the new version.
import importlib
import utils
importlib.reload(utils)
import dashboard_mrms
importlib.reload(dashboard_mrms)
from dashboard_mrms import MRMSTransposeDashboard, DEFAULT_NC, DEFAULT_PARQUET

import panel as pn
import holoviews as hv
pn.extension("bokeh")
hv.extension("bokeh")

# ── Configure paths here ──────────────────────────────────────────────────
NC_FILE  = DEFAULT_NC       # ← swap to any other MRMS NetCDF path if needed
SHP_FILE = DEFAULT_PARQUET  # ← set to None to disable watershed overlay
PORT     = 5006
# ─────────────────────────────────────────────────────────────────────────

dashboard = MRMSTransposeDashboard(NC_FILE, shp_path=SHP_FILE)

def _serve():
    pn.serve(dashboard.layout(), port=PORT, show=False, verbose=False)

_server_thread = threading.Thread(target=_serve, daemon=True)
_server_thread.start()

time.sleep(2.0)   # give the server a moment to start (watersheds take a bit longer)

url = f"http://localhost:{PORT}"
print(f"Dashboard running at: {url}")
webbrowser.open(url)


C:\Users\domenicbrooks\OneDrive - Research Triangle Institute\CIROH Flash Flooding\code\dashboard_mrms.py:64: UserWarning: Using Panel interactively in VSCode notebooks requires the jupyter_bokeh package to be installed. You can install it with:

   pip install jupyter_bokeh

or:
    conda install jupyter_bokeh

and try again.
  pn.extension("bokeh")


Loading watersheds…
  Loaded 50 watersheds
  Loaded coastline
  Loaded 50 gauge locations
Fetching rating curves for 50 basins…


  Rating curves loaded: 50/50


C:\Users\domenicbrooks\OneDrive - Research Triangle Institute\CIROH Flash Flooding\code\dashboard_mrms.py:64: UserWarning: Using Panel interactively in VSCode notebooks requires the jupyter_bokeh package to be installed. You can install it with:

   pip install jupyter_bokeh

or:
    conda install jupyter_bokeh

and try again.
  pn.extension("bokeh")


Loading watersheds…
  Loaded 50 watersheds
  Loaded coastline
  Loaded 50 gauge locations
Fetching rating curves for 50 basins…
  Rating curves loaded: 50/50


C:\Users\domenicbrooks\AppData\Local\Temp\ipykernel_12172\1364930366.py:20: UserWarning: Using Panel interactively in VSCode notebooks requires the jupyter_bokeh package to be installed. You can install it with:

   pip install jupyter_bokeh

or:
    conda install jupyter_bokeh

and try again.
  pn.extension("bokeh")


Loading watersheds…
  Loaded 50 watersheds
  Loaded coastline
  Loaded 50 gauge locations
Fetching rating curves for 50 basins…
  Rating curves loaded: 50/50
Dashboard running at: http://localhost:5006


True

In [3]:
# ── Rating curve debug ────────────────────────────────────────────────────────
# Run this cell to inspect the raw NWIS response and parsed rating curve
# for any calibrated gauge.  Useful if the stage plot looks wrong.

import requests
from dashboard_mrms import _AVAILABLE_BASIN_IDS

INSPECT_BASIN = "16103000"   # ← change to any basin ID

url = f"https://waterdata.usgs.gov/nwisweb/get_ratings?file_type=exsa&site_no={INSPECT_BASIN}"
resp = requests.get(url, timeout=15)
print(f"HTTP status : {resp.status_code}")
print(f"Content-Type: {resp.headers.get('Content-Type', 'unknown')}")
print(f"\n── First 30 lines of raw response ──────────────────────────────")
for i, line in enumerate(resp.text.splitlines()[:30]):
    print(f"  {i:02d}: {repr(line)}")

# Show the parsed rating curve from the dashboard's cache
rc = dashboard._rating_curves.get(INSPECT_BASIN)
if rc is None:
    print(f"\n⚠  No rating curve cached for {INSPECT_BASIN}")
else:
    import numpy as np
    print(f"\n── Parsed rating curve ({len(rc['discharge_cfs'])} points) ─────────────────")
    print(f"  Discharge range : {rc['discharge_cfs'].min():.3f} – {rc['discharge_cfs'].max():.3f} cfs")
    print(f"  Stage range     : {rc['stage_ft'].min():.3f} – {rc['stage_ft'].max():.3f} ft")
    print(f"\n  First 5 rows (discharge cfs, stage ft):")
    for q, h in zip(rc['discharge_cfs'][:5], rc['stage_ft'][:5]):
        print(f"    {q:10.3f} cfs  →  {h:.3f} ft")
    print(f"\n  Last 5 rows:")
    for q, h in zip(rc['discharge_cfs'][-5:], rc['stage_ft'][-5:]):
        print(f"    {q:10.3f} cfs  →  {h:.3f} ft")


HTTP status : 200
Content-Type: text/plain; charset=UTF-8

── First 30 lines of raw response ──────────────────────────────
  00: '# //UNITED STATES GEOLOGICAL SURVEY       http://water.usgs.gov/'
  01: '# //NATIONAL WATER INFORMATION SYSTEM     http://water.usgs.gov/data.html'
  02: '# //DATA ARE PROVISIONAL AND SUBJECT TO CHANGE UNTIL PUBLISHED BY USGS'
  03: '# //RETRIEVED: 2024-12-06 22:05:02'
  04: '# //WARNING'
  05: '# //WARNING The stage-discharge rating provided in this file should be'
  06: '# //WARNING considered provisional and subject to change. Stage-discharge'
  07: '# //WARNING ratings change over time as the channel features that control'
  08: '# //WARNING the relation between stage and discharge vary. Users are'
  09: '# //WARNING cautioned to consider carefully the applicability of this'
  10: '# //WARNING rating before using it for decisions that concern personal or'
  11: '# //WARNING public safety or operational consequences.'
  12: '# //FILE TYPE="NWIS RATING" '

In [2]:
# ── Rating curve availability summary ────────────────────────────────────────
# Shows which calibrated basins have a usable rating curve and which don't.

from dashboard_mrms import _AVAILABLE_BASIN_IDS

ok, missing = [], []
for basin_id in _AVAILABLE_BASIN_IDS:
    rc = dashboard._rating_curves.get(basin_id)
    if rc is not None:
        q_max = rc['discharge_cfs'].max()
        h_max = rc['stage_ft'].max()
        ok.append((basin_id, len(rc['discharge_cfs']), q_max, h_max))
    else:
        missing.append(basin_id)

print(f"✓  {len(ok)} / {len(_AVAILABLE_BASIN_IDS)} basins have a rating curve\n")

if ok:
    print(f"  {'Basin ID':<12} {'Points':>6}  {'Max Q (cfs)':>12}  {'Max Stage (ft)':>14}")
    print("  " + "-" * 50)
    for basin_id, n, q_max, h_max in ok:
        print(f"  {basin_id:<12} {n:>6}  {q_max:>12.1f}  {h_max:>14.2f}")

if missing:
    print(f"\n✗  {len(missing)} basin(s) with no rating curve:")
    for basin_id in missing:
        print(f"  {basin_id}")
else:
    print("\n  All calibrated basins have rating curves.")


✓  50 / 50 basins have a rating curve

  Basin ID     Points   Max Q (cfs)  Max Stage (ft)
  --------------------------------------------------
  16010000       1425       11800.0           16.00
  16019000        777        4500.0            8.50
  16060000       2163       68800.0           23.10
  16068000       1211       12541.6           11.80
  16071500        604         870.0            6.70
  16097500        959        4590.0           10.19
  16103000       1898       31342.9           17.98
  16108000       1038       19997.5           10.91
  16200000       1337        6600.0           14.40
  16208000       1185        5016.0           12.87
  16211600        884        3000.0           10.05
  16212490        726        2700.0           12.25
  16212601       1602        1750.0           20.93
  16226200       1143        3610.7           17.34
  16227500       1206        4733.5           14.22
  16229000       1165        8000.0           13.50
  16238000        643   

In [4]:
"""
Sanity check: verify that each calibrated basin maps to the correct
parameter file, and that the values in that file match the best_params CSV
produced by the calibration optimiser.

Also flags any basins where the _cal JSON is missing or whose parameters
deviate unexpectedly from the CSV.
"""
import json
import pandas as pd
from pathlib import Path
from dashboard_mrms import (
    HAWAII_RESULTS_DIR, HYDRO_MODEL_CONFIG_DIR,
    _AVAILABLE_BASIN_IDS, WATERSHED_NAME_COL,
)

# ── 1. Show the name → gauge-ID mapping from the running dashboard ────────
print(f"{'Watershed name':<55} {'Gauge ID'}")
print("-" * 70)
for name, gid in sorted(dashboard._basin_id_for_name.items()):
    print(f"{name:<55} {gid}")

# ── 2. For each basin: load _cal JSON, compare against best_params CSV ────
print("\n\n=== Parameter validation (cal JSON vs. best_params CSV) ===\n")

SAC_KEYS_OPTIMISED = ["uzfwm", "lzfpm", "lzfsm", "lzpk", "lzsk",
                       "adimp", "zperc", "rexp"]   # keys written to best_params CSV

results = []
for basin in _AVAILABLE_BASIN_IDS:
    cal_path = HAWAII_RESULTS_DIR / f"configurations/sac/sac_params_{basin}_cal.json"
    csv_path = HAWAII_RESULTS_DIR / f"output/basin_{basin}/best_params_{basin}_kge_q90.csv"

    if not cal_path.exists():
        results.append({"basin": basin, "status": "MISSING _cal JSON", "max_diff": None})
        continue
    if not csv_path.exists():
        results.append({"basin": basin, "status": "MISSING best_params CSV", "max_diff": None})
        continue

    cal  = json.loads(cal_path.read_text())
    bpdf = pd.read_csv(csv_path).set_index("parameter")["best_value"]

    diffs = {}
    for k in SAC_KEYS_OPTIMISED:
        if k not in bpdf.index:
            continue
        cal_val = cal[k][0] if isinstance(cal[k], list) else cal[k]
        diffs[k] = abs(cal_val - bpdf[k])

    max_diff = max(diffs.values()) if diffs else 0.0
    status = "OK" if max_diff < 1e-4 else f"MISMATCH (max_diff={max_diff:.6f})"
    results.append({"basin": basin, "status": status, "max_diff": max_diff})

summary = pd.DataFrame(results)
pd.set_option("display.max_rows", 60)
print(summary.to_string(index=False))

ok  = (summary["status"] == "OK").sum()
bad = len(summary) - ok
print(f"\n{ok}/{len(summary)} basins passed  |  {bad} issue(s)")


Watershed name                                          Gauge ID
----------------------------------------------------------------------
Alakahi Stream near Kamuela, HI                         16725000
EB of NF Wailua River nr Lihue, Kauai, HI               16068000
Halaulani Str at alt 400 ft nr Kilauea, Kauai, HI       16097500
Halawa Stream near Halawa, Molokai, HI                  16400000
Hanalei River nr Hanalei, Kauai, HI                     16103000
Hanawi Stream near Nahiku, Maui, HI                     16508000
Heeia Stream at Haiku Valley nr Kaneohe, Oahu, HI       16275000
Honokohau Stream near Honokohau, Maui, HI               16620000
Honolii Stream nr Papaikou, HI                          16717000
Honopou Stream near Huelo, Maui, HI                     16587000
Honouliuli Str at H-1 Freeway nr Waipahu, Oahu, HI      16212490
Kahakuloa Stream near Honokohau, Maui, HI               16618000
Kahaluu Str nr Ahuimanu, Oahu, HI                       16283200
Kahana Str at alt 3

In [15]:
import pandas as pd

In [17]:
dashboard._basin_id_for_name

{'Kawaikoi Stream nr Waimea, Kauai, HI': '16010000',
 'Waialae Str at alt 3,820 ft nr Waimea, Kauai, HI': '16019000',
 'SF Wailua River nr Lihue, Kauai, HI': '16060000',
 'EB of NF Wailua River nr Lihue, Kauai, HI': '16068000',
 'Left Branch Opaekaa Str nr Kapaa, Kauai, HI': '16071500',
 'Halaulani Str at alt 400 ft nr Kilauea, Kauai, HI': '16097500',
 'Hanalei River nr Hanalei, Kauai, HI': '16103000',
 'Wainiha River nr Hanalei, Kauai, HI': '16108000',
 'NF Kaukonahua Str abv RB, nr Wahiawa, Oahu, HI': '16200000',
 'SF Kaukonahua Str at E pump, nr Wahiawa, Oahu, HI': '16208000',
 'Makaha Str nr Makaha, Oahu, HI': '16211600',
 'Honouliuli Str at H-1 Freeway nr Waipahu, Oahu, HI': '16212490',
 'Waikele Str at Wheeler Field, Oahu, HI': '16212601',
 'N. Halawa Str nr Honolulu, Oahu, HI': '16226200',
 'Moanalua Stream nr Kaneohe, Oahu, HI': '16227500',
 'Kalihi Str nr Honolulu, Oahu, HI': '16229000',
 'Makiki Stream at King St. bridge, Oahu, HI': '16238000',
 'Waihi Stream at Honolulu, Oah